# بررسی بازار فیزیکی گندله سنگ‌آهن

این نوت‌بوک فقط معاملات **نقدی** و **نقدی (مچینگ)** با قیمت و مقدار مثبت را بررسی می‌کند. داده خام تغییر نمی‌کند. معیار قیمت، ستون `Price` منبع است و رابطه `Price = TotalPrice / Quantity` به‌صورت مستقل کنترل می‌شود.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)

RAW_CSV = Path("../data/raw/physical/pellet_physical_raw.csv")
if not RAW_CSV.exists():
    raise FileNotFoundError(f"Run this notebook from {Path.cwd()} or correct RAW_CSV: {RAW_CSV}")

raw = pd.read_csv(RAW_CSV, encoding="utf-8-sig", low_memory=False)
numeric_columns = ["Price", "Quantity", "TotalPrice", "MinPrice", "MaxPrice", "arze", "taghaza"]
for column in numeric_columns:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

raw["date_jalali"] = raw["date"].astype(str).str.replace("-", "/", regex=False)
cash_contracts = ["نقدی", "نقدی (مچینگ)"]
cash = raw.loc[
    raw["ContractType"].isin(cash_contracts)
    & raw["Quantity"].gt(0)
    & raw["Price"].gt(0)
].copy()

print(f"Raw rows: {len(raw):,}")
print(f"Positive cash/cash-matching rows: {len(cash):,}")
print(f"Dates: {cash['date_jalali'].nunique():,}")
print(f"Traded symbols: {cash['Symbol'].nunique():,}")


## ۱. اعتبارسنجی قیمت

`Price` قیمت گزارش‌شده بورس کالاست. نسبت `TotalPrice / Quantity` صرفاً کنترل مستقل آن است. اختلاف حداکثر ۰٫۵ ریال می‌تواند از گردکردن قیمت ناشی شود.


In [ ]:
cash["price_from_value_quantity"] = cash["TotalPrice"] / cash["Quantity"]
cash["price_validation_abs_diff"] = (cash["Price"] - cash["price_from_value_quantity"]).abs()

price_validation = pd.Series({
    "rows": len(cash),
    "missing_total_price": int(cash["TotalPrice"].isna().sum()),
    "exact_matches": int(cash["price_validation_abs_diff"].eq(0).sum()),
    "maximum_absolute_difference": cash["price_validation_abs_diff"].max(),
    "maximum_relative_difference_pct": (100 * cash["price_validation_abs_diff"] / cash["Price"]).max(),
})
display(price_validation.to_frame("value"))

if cash["price_validation_abs_diff"].max() > 0.51:
    raise ValueError("Price no longer agrees with TotalPrice / Quantity within rounding tolerance.")


## ۲. شناسایی شرکت‌ها و نمادها

نماد (`Symbol`) شناسه پایدار شرکت در تحلیل است؛ زیرا نام بعضی تولیدکنندگان در طول تاریخ تغییر نگارشی داشته است.


In [ ]:
company_profile = (
    cash.groupby("Symbol", as_index=False)
    .agg(
        producer_names=("ProducerName", lambda values: " | ".join(sorted(set(values.dropna().astype(str))))),
        first_date=("date_jalali", "min"),
        last_date=("date_jalali", "max"),
        trade_days=("date_jalali", "nunique"),
        rows=("Quantity", "size"),
        volume=("Quantity", "sum"),
    )
    .sort_values("volume", ascending=False)
)
company_profile["volume_share_pct"] = 100 * company_profile["volume"] / company_profile["volume"].sum()
display(company_profile)


## ۳. قیمت روزانه هر شرکت

ابتدا قیمت موزون هر شرکت در هر روز و برای هر نوع قرارداد جداگانه ساخته می‌شود. سپس یک قیمت ترکیبی نقدی+مچینگ نیز برای مقایسه شرکت‌ها محاسبه می‌شود. موزون‌سازی فقط بین ردیف‌های همان شرکت و همان روز انجام می‌شود.


In [ ]:
cash["price_x_quantity"] = cash["Price"] * cash["Quantity"]

daily_company_contract = (
    cash.groupby(["date_jalali", "Symbol", "ContractType"], as_index=False)
    .agg(
        price_x_quantity=("price_x_quantity", "sum"),
        quantity=("Quantity", "sum"),
        rows=("Quantity", "size"),
        producer_name=("ProducerName", "last"),
    )
)
daily_company_contract["weighted_price"] = (
    daily_company_contract["price_x_quantity"] / daily_company_contract["quantity"]
)

daily_company = (
    cash.groupby(["date_jalali", "Symbol"], as_index=False)
    .agg(
        price_x_quantity=("price_x_quantity", "sum"),
        quantity=("Quantity", "sum"),
        rows=("Quantity", "size"),
        producer_name=("ProducerName", "last"),
        contract_count=("ContractType", "nunique"),
    )
)
daily_company["weighted_price"] = daily_company["price_x_quantity"] / daily_company["quantity"]

display(daily_company_contract.head())
display(daily_company.head())


## ۴. مقایسه نقدی و نقدی‌مچینگ برای یک شرکت در یک روز


In [ ]:
cash_vs_matching = (
    daily_company_contract.pivot_table(
        index=["date_jalali", "Symbol"],
        columns="ContractType",
        values="weighted_price",
        aggfunc="first",
    )
    .dropna(subset=cash_contracts)
    .reset_index()
)
cash_vs_matching["matching_vs_cash_pct"] = 100 * (
    cash_vs_matching["نقدی (مچینگ)"] / cash_vs_matching["نقدی"] - 1
)
display(cash_vs_matching["matching_vs_cash_pct"].describe())
display(cash_vs_matching.reindex(cash_vs_matching["matching_vs_cash_pct"].abs().sort_values(ascending=False).index).head(20))


## ۵. نمودار قیمت شرکت‌های اصلی

پارامتر `TOP_N` را برای کم یا زیادکردن تعداد شرکت‌ها تغییر دهید.


In [ ]:
TOP_N = 4
top_symbols = company_profile.head(TOP_N)["Symbol"].tolist()
plot_data = daily_company.loc[daily_company["Symbol"].isin(top_symbols)].copy()
price_wide = plot_data.pivot(index="date_jalali", columns="Symbol", values="weighted_price").sort_index()

ax = price_wide.plot(figsize=(16, 7), linewidth=1.5)
ax.set_title("قیمت روزانه موزون نقدی و نقدی‌مچینگ شرکت‌های اصلی گندله")
ax.set_xlabel("تاریخ جلالی")
ax.set_ylabel("Price گزارش‌شده بورس کالا")
ax.legend(title="نماد", ncol=2)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## ۶. فاصله قیمت هر شرکت از قیمت موزون کل بازار همان روز

این تبدیل اثر روند عمومی قیمت را کم می‌کند و برای تشخیص premium یا discount پایدار شرکت‌ها مناسب‌تر است.


In [ ]:
daily_market = (
    daily_company.groupby("date_jalali", as_index=False)
    .agg(price_x_quantity=("price_x_quantity", "sum"), quantity=("quantity", "sum"))
)
daily_market["market_weighted_price"] = daily_market["price_x_quantity"] / daily_market["quantity"]
comparison = daily_company.merge(daily_market[["date_jalali", "market_weighted_price"]], on="date_jalali", how="left")
comparison["company_vs_market_pct"] = 100 * (comparison["weighted_price"] / comparison["market_weighted_price"] - 1)

premium_summary = (
    comparison.groupby("Symbol")["company_vs_market_pct"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .join(company_profile.set_index("Symbol")[["volume_share_pct"]])
    .sort_values("volume_share_pct", ascending=False)
)
display(premium_summary)

box_symbols = company_profile.head(10)["Symbol"]
box_data = [comparison.loc[comparison["Symbol"].eq(symbol), "company_vs_market_pct"].dropna() for symbol in box_symbols]
fig, ax = plt.subplots(figsize=(15, 6))
ax.boxplot(box_data, tick_labels=box_symbols, showfliers=False)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("توزیع فاصله قیمت شرکت‌ها از قیمت موزون همان روز بازار")
ax.set_ylabel("درصد فاصله از بازار")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## ۷. مقایسه مقطعی در یک روز دلخواه

تاریخ را از میان تاریخ‌های دارای چند شرکت انتخاب کنید.


In [ ]:
available_dates = (
    daily_company.groupby("date_jalali")["Symbol"].nunique().loc[lambda s: s >= 2].index.tolist()
)
SELECTED_DATE = available_dates[-1]
selected = daily_company.loc[daily_company["date_jalali"].eq(SELECTED_DATE)].sort_values("weighted_price")
display(selected[["date_jalali", "Symbol", "producer_name", "weighted_price", "quantity", "contract_count"]])

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(selected["Symbol"], selected["weighted_price"])
ax.set_title(f"مقایسه قیمت شرکت‌های گندله در {SELECTED_DATE}")
ax.set_xlabel("قیمت موزون نقدی و نقدی‌مچینگ")
plt.tight_layout()
plt.show()


## ۸. خروجی تشخیصی برای تصمیم‌گیری

این جدول‌ها در حافظه ساخته می‌شوند و چیزی را در raw بازنویسی نمی‌کنند. پس از مشاهده نمودارها درباره ترکیب مناسب شرکت‌ها برای underlying تصمیم می‌گیریم.


In [ ]:
daily_dispersion = (
    daily_company.groupby("date_jalali")
    .agg(
        company_count=("Symbol", "nunique"),
        minimum_price=("weighted_price", "min"),
        maximum_price=("weighted_price", "max"),
        total_quantity=("quantity", "sum"),
    )
    .reset_index()
)
daily_dispersion["max_min_spread_pct"] = 100 * (
    daily_dispersion["maximum_price"] / daily_dispersion["minimum_price"] - 1
)
display(daily_dispersion.describe())
display(daily_dispersion.sort_values("max_min_spread_pct", ascending=False).head(20))
